# LinkedIn Ads — daily report

Generates the daily Campaign Manager report for the promo-credit ad set:
budget pace, spend/clicks/impressions trends, and a plain-language summary.
Executed by Celery (`linkedin-ads-daily-report` beat) via papermill inside
`social-worker-default` — the kernel inherits the worker env (`DATABASE_URL`).
Editable here in Jupyter; the worker re-runs the saved file.

In [ ]:
# papermill parameters — do not add code before this cell
campaign_id = ""
lookback_days = 45
is_final = False
end_date = ""        # ISO date the campaign schedule ends
status_hint = ""     # live status from collect_metrics (draft/active/paused)


In [ ]:
import html as _html
import json
from datetime import date, datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import select

from app.core.config import get_settings
from app.db.session import async_session_maker
from app.models.linkedin_ads import AdCampaignSnapshot

settings = get_settings()
tz = ZoneInfo(settings.APP_TIMEZONE)
today = datetime.now(tz).date()
out = Path(output_dir)
end = date.fromisoformat(end_date) if end_date else None


In [ ]:
async def _load_history() -> pd.DataFrame:
    async with async_session_maker() as db:
        rows = (
            await db.execute(
                select(AdCampaignSnapshot)
                .where(AdCampaignSnapshot.campaign_id == campaign_id)
                .order_by(AdCampaignSnapshot.captured_at)
            )
        ).scalars().all()
    return pd.DataFrame([
        {
            "captured_at": r.captured_at,
            "spend_eur": float(r.spend_eur or 0),
            "impressions": int(r.impressions or 0),
            "clicks": int(r.clicks or 0),
            "engagements": int(r.engagements or 0),
            "ctr": float(r.ctr or 0),
            "cpc_eur": float(r.cpc_eur or 0),
            "engagement_rate": float(r.engagement_rate or 0),
            "budget_eur": float(r.budget_eur or 0),
            "status": r.status or "unknown",
            "campaign_name": r.campaign_name or "",
        }
        for r in rows
    ])

hist = await _load_history()
if hist.empty:
    raise RuntimeError(f"no snapshots for campaign {campaign_id!r}")
latest = hist.iloc[-1]
prev = hist.iloc[-2] if len(hist) > 1 else None
status = status_hint or latest["status"]
name = latest["campaign_name"] or "Cloudless boost - Sep 2026 - coupon"
print(f"{len(hist)} snapshots, latest spend EUR {latest['spend_eur']:.2f}, status={status}")


## Charts — spend pace + daily performance

In [ ]:
TEAL, VOID, CYAN = "#0a7785", "#0a0a0f", "#00d4c8"

def _style(fig, ax):
    fig.patch.set_facecolor(VOID)
    ax.set_facecolor(VOID)
    ax.tick_params(colors="#c9d1d9", labelsize=9)
    for s in ax.spines.values():
        s.set_color("#30363d")
    ax.grid(True, color="#21262d", linewidth=0.6)
    ax.title.set_color("#e6edf3")

hist_d = hist.copy()
hist_d["day"] = pd.to_datetime(hist_d["captured_at"]).dt.date
daily = hist_d.groupby("day").agg(
    spend=("spend_eur", "max"), imp=("impressions", "max"),
    clicks=("clicks", "max"), eng=("engagements", "max"),
).tail(lookback_days)

# 1) Cumulative spend vs budget cap
spend_png = out / f"linkedin_ads_spend-{run_id}.png"
fig, ax = plt.subplots(figsize=(7, 2.8), dpi=140)
ax.plot(daily.index, daily["spend"], color=CYAN, lw=2, marker="o", ms=3, label="spend (EUR)")
if latest["budget_eur"]:
    ax.axhline(latest["budget_eur"], color="#ff7b72", ls="--", lw=1.4,
               label=f"lifetime budget EUR {latest['budget_eur']:.0f}")
ax.set_title("Cumulative spend vs lifetime budget", fontsize=11)
ax.legend(facecolor=VOID, labelcolor="#c9d1d9", fontsize=8, framealpha=0)
fig.autofmt_xdate()
_style(fig, ax)
fig.tight_layout()
fig.savefig(spend_png, facecolor=VOID)
plt.close(fig)

# 2) Daily clicks + impressions (per-day max delta)
d_imp = daily["imp"].diff().clip(lower=0)
d_clicks = daily["clicks"].diff().clip(lower=0)
perf_png = out / f"linkedin_ads_perf-{run_id}.png"
fig, ax = plt.subplots(figsize=(7, 2.8), dpi=140)
ax2 = ax.twinx()
ax.bar(daily.index, d_imp, color="#1f6f8b", width=0.7, label="impressions")
ax2.plot(daily.index, d_clicks, color=CYAN, lw=1.8, marker="o", ms=3, label="clicks")
ax.set_title("Daily impressions (bars) + clicks (line)", fontsize=11)
ax2.tick_params(colors="#c9d1d9", labelsize=9)
ax2.grid(False)
_style(fig, ax)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(perf_png, facecolor=VOID)
plt.close(fig)
print(spend_png.name, perf_png.name)


## Report body — plain-language text + HTML

In [ ]:
def _eur(v: float) -> str:
    return f"EUR {v:,.2f}".rstrip("0").rstrip(".") if v else "EUR 0"

status_label = {
    "active": "[Running]", "paused": "[Paused]",
    "draft": "[Draft — not live yet]", "completed": "[Finished]",
}.get(status, str(status).title())

lines = [f"*LinkedIn ad report — {name}* ({today.isoformat()})", ""]
lines.append(f"*Status:* {status_label}")

budget = latest["budget_eur"]
if budget:
    pct = latest["spend_eur"] / budget * 100
    lines.append(f"*Spend:* {_eur(latest['spend_eur'])} of {_eur(budget)} lifetime budget ({pct:.0f}% used)")
else:
    lines.append(f"*Spend so far:* {_eur(latest['spend_eur'])}")

if prev is not None:
    bits = []
    d_spend = latest["spend_eur"] - prev["spend_eur"]
    d_clicks = int(latest["clicks"] - prev["clicks"])
    d_eng = int(latest["engagements"] - prev["engagements"])
    d_imp = int(latest["impressions"] - prev["impressions"])
    if d_spend > 0: bits.append(f"+{_eur(d_spend)} spend")
    if d_imp > 0: bits.append(f"+{d_imp:,} impressions")
    if d_clicks > 0: bits.append(f"+{d_clicks} clicks")
    if d_eng > 0: bits.append(f"+{d_eng} engagements")
    if bits:
        lines.append(f"*Since last report:* {', '.join(bits)}")

stats = []
if latest["clicks"]: stats.append(f"{int(latest['clicks'])} clicks")
if latest["cpc_eur"]: stats.append(f"CPC {_eur(latest['cpc_eur'])}")
if latest["ctr"]: stats.append(f"CTR {latest['ctr']:.2f}%")
if latest["engagements"]: stats.append(f"{int(latest['engagements'])} engagements")
if latest["engagement_rate"]: stats.append(f"ER {latest['engagement_rate']:.2f}%")
if latest["impressions"]: stats.append(f"~{int(latest['impressions']):,} impressions")
if stats:
    lines.append("*Totals:* " + " · ".join(stats))

lines.append("")
if end and budget:
    days_left = (end - today).days
    if status == "active" and days_left > 0 and latest["spend_eur"]:
        cap = pd.to_datetime(hist["captured_at"]).iloc[0].date()
        daily_avg = latest["spend_eur"] / max(1, (today - cap).days or 1)
        projected = latest["spend_eur"] + daily_avg * days_left
        verdict = "on track" if projected <= budget * 1.05 else "running hot"
        lines.append(f"*Pace:* {verdict} — projecting ~{_eur(projected)} by {end.isoformat()} vs {_eur(budget)} cap.")
    elif status == "draft":
        lines.append("*Note:* the ad set is still a draft — it isn't spending. "
                     "Launch it in Campaign Manager when the creative is approved.")
lines.append("*Card safety:* spend stays inside the promo credit — the Visa on file is not expected to be charged.")
if end:
    lines.append(f"Campaign ends {end.isoformat()}. Report again tomorrow 10:00.")
if is_final:
    lines += ["", "*Final report* — the campaign schedule has ended."]

text = "\n".join(lines)


In [ ]:
# Organic page section (best-effort, reuses the service builder)
try:
    from app.services.linkedin_ads_report import build_org_section
    async def _org():
        async with async_session_maker() as db:
            return await build_org_section(db)
    org_section = await _org()
    if org_section:
        text += org_section
except Exception as exc:  # noqa: BLE001 — organic section is best-effort
    print("org section skipped:", exc)
    org_section = ""


In [ ]:
def _mrkdwn_to_html(t: str) -> str:
    rows = []
    for ln in t.split("\n"):
        esc = _html.escape(ln).replace("*", "")
        if not esc.strip():
            rows.append("<div style='height:8px'></div>")
        elif esc.startswith(("Status:", "Spend:", "Since", "Totals:", "Pace:", "Note:")):
            rows.append(f"<p style='margin:2px 0'>{esc}</p>")
        else:
            rows.append(f"<p style='margin:6px 0;font-weight:600'>{esc}</p>")
    return "".join(rows)

html_body = f"""<!doctype html><html><body style="margin:0;background:#0a0a0f;padding:24px;
font-family:-apple-system,Segoe UI,Roboto,sans-serif;color:#e6edf3">
<div style="max-width:640px;margin:0 auto">
{_mrkdwn_to_html(text)}
<div style="margin-top:16px">
<img src="cid:spend" style="width:100%;border-radius:8px;border:1px solid #30363d"/>
<img src="cid:perf" style="width:100%;margin-top:10px;border-radius:8px;border:1px solid #30363d"/>
</div>
<p style="color:#8b949e;font-size:12px;margin-top:16px">
Generated by the SocialAuto notebook reporting pipeline —
source: Campaign Manager via LinkedIn sidecar · ad_campaign_snapshots
</p></div></body></html>"""

text_path = out / f"linkedin_ads-{run_id}.txt"
html_path = out / f"linkedin_ads-{run_id}.html"
text_path.write_text(text)
html_path.write_text(html_body)
print(text_path.name, html_path.name)


In [ ]:
manifest = {
    "subject": f"LinkedIn ad report — {today.isoformat()}",
    "text_file": text_path.name,
    "html_file": html_path.name,
    "attachments": [
        {"file": spend_png.name, "cid": "spend", "mime": "image/png"},
        {"file": perf_png.name, "cid": "perf", "mime": "image/png"},
    ],
    "extra": {"campaign_id": campaign_id, "status": status},
}
Path(manifest_path).write_text(json.dumps(manifest))
print("manifest ->", manifest_path)
